# NB134: The Window-Zero Exponent

## Investigation: Is x_eff(w0, lepton) = p₂ = 3 a real identity?

NB133 discovered that the effective exponent at window 0 for leptons is 3.000376 ≈ p₂ = 3.
This means m_μ/m_e ≈ C₀_lep³ where C₀ is the window-0 CP ratio.

**Questions**:
1. Is this T-independent? (Does x_eff converge to exactly 3 as T → ∞?)
2. What is the quark counterpart? Is x_eff(w0, quark) recognizable?
3. Does x = p₂ appear at other mass channels?
4. Can we derive x = 3 from the CP ratio structure algebraically?

**Why this matters**: If confirmed, this would be a simpler mass formula than the cumulative
pipeline, working directly at the T-independent window-0 level. The chirality prime p₂ = 3
appearing as the exponent connects mass to the three degrees of reception.

In [2]:
# Setup
import sys, numpy as np, time
from pathlib import Path
from fractions import Fraction

ROOT = Path.cwd().parent
if str(ROOT / 'scripts') not in sys.path:
    sys.path.insert(0, str(ROOT / 'scripts'))

from solenoid_algebra import (SA, RHO, KAPPA, EPSILON, OMEGA,
                               X4, X3, X2, LAM7, X4_LEP,
                               PHYSICAL_CROSSINGS, CP_PAIRS, SM_TARGETS,
                               ACTIVE_PRIMES)
from solenoid_system import SolenoidSystem

primes = [2, 3, 5, 7]
p1, p2, p3, p4 = primes
P4 = 210
gamma = np.array([p**2 for p in primes])

target_q = SM_TARGETS['m_s/m_d'][0]   # ~20.0
target_l = SM_TARGETS['m_mu/m_e'][0]  # ~206.768

# Physical crossing indices
physical_cis = {name: info['ci'] for name, info in PHYSICAL_CROSSINGS.items()}
print(f'Physical crossings: {physical_cis}')
print(f'Targets: m_s/m_d = {target_q}, m_μ/m_e = {target_l}')
print(f'Algebraic exponents: X₄ = {X4:.6f}, X₄_LEP = {X4_LEP:.6f}')
print(f'p₂ = {p2}')

Physical crossings: {'QUARK_g1': 11, 'LEPTON_g1': 31, 'LEPTON_g2': 61, 'QUARK_g2': 191}
Targets: m_s/m_d = 20.0, m_μ/m_e = 206.768
Algebraic exponents: X₄ = 7.639437, X₄_LEP = 7.798592
p₂ = 3


## S1: T-Independence Test

Run the cascade at T = 500, 1000, 2000, 5000, 10000.
At each T, compute the window-0 CP ratio and the effective exponent.
If x_eff → 3 as T → ∞, this is a real identity.

In [3]:
# S1: T-sweep for window-0 exponent

from solenoid_jax import integrate_all_branches_jax

sys0 = SolenoidSystem(primes)
branches = sys0.all_branches()

T_values = [500, 1000, 2000, 5000, 10000]
results_by_T = {}

print('S1: T-INDEPENDENCE TEST')
print('=' * 70)
print()

for T_max in T_values:
    t0 = time.time()
    # Evaluate at coprime crossings up to T_max
    ci_all = SA.coprime_indices(T_max)
    t_eval = ci_all.astype(float)
    
    res = integrate_all_branches_jax(branches, t_eval, float(T_max),
                                     kappa=KAPPA, epsilon=EPSILON, omega=OMEGA)
    
    # Stack and wrap
    all_R = np.stack([res[br] for br in branches])  # (210, n_ci, 4)
    all_R_w = np.mod(all_R + np.pi, 2*np.pi) - np.pi  # wrap to [-π,π]
    
    # Get CRT labels
    ci_a3, ci_a5, ci_a7 = SA.sector_labels(ci_all)
    
    # Filter a5=0 only
    mask_a5 = ci_a5 == 0
    R3_w = all_R_w[:, mask_a5, 3]  # level 3, a5=0
    ci_filt = ci_all[mask_a5]
    a3_filt = ci_a3[mask_a5]
    a7_filt = ci_a7[mask_a5]
    
    # Window-0: first primorial window [1, P₄)
    w0_mask = ci_filt < P4
    R3_w0 = R3_w[:, w0_mask]
    a3_w0 = a3_filt[w0_mask]
    a7_w0 = a7_filt[w0_mask]
    ci_w0 = ci_filt[w0_mask]
    
    # Lepton CP at window 0: sectors (a3=0, a7=1) vs (a3=0, a7=5)
    lep_g1 = (a3_w0 == 0) & (a7_w0 == 1)
    lep_g2 = (a3_w0 == 0) & (a7_w0 == 5)
    quark_g1 = (a3_w0 == 1) & (a7_w0 == 4)
    quark_g2 = (a3_w0 == 1) & (a7_w0 == 2)
    
    rms_l_g1 = np.sqrt(np.mean(R3_w0[:, lep_g1]**2))
    rms_l_g2 = np.sqrt(np.mean(R3_w0[:, lep_g2]**2))
    rms_q_g1 = np.sqrt(np.mean(R3_w0[:, quark_g1]**2))
    rms_q_g2 = np.sqrt(np.mean(R3_w0[:, quark_g2]**2))
    
    C0_l = rms_l_g1 / rms_l_g2
    C0_q = rms_q_g1 / rms_q_g2
    
    x_l = np.log(target_l) / np.log(C0_l)
    x_q = np.log(target_q) / np.log(C0_q)
    
    elapsed = time.time() - t0
    
    results_by_T[T_max] = {
        'C0_l': C0_l, 'C0_q': C0_q,
        'x_l': x_l, 'x_q': x_q,
        'n_ci': len(ci_all)
    }
    
    print(f'T={T_max:6d}  ({elapsed:.1f}s, {len(ci_all)} crossings)')
    print(f'  Lepton:  C₀ = {C0_l:.6f}  x_eff = {x_l:.6f}  dev from 3: {(x_l/3-1)*100:+.4f}%')
    print(f'  Quark:   C₀ = {C0_q:.6f}  x_eff = {x_q:.6f}')
    print()

# Convergence analysis
print('CONVERGENCE:')
x_l_vals = [results_by_T[T]['x_l'] for T in T_values]
x_q_vals = [results_by_T[T]['x_q'] for T in T_values]
print(f'  Lepton x_eff range: [{min(x_l_vals):.6f}, {max(x_l_vals):.6f}]')
print(f'  Lepton x_eff spread: {max(x_l_vals) - min(x_l_vals):.6f}')
print(f'  Quark x_eff range: [{min(x_q_vals):.6f}, {max(x_q_vals):.6f}]')
print(f'  Quark x_eff spread: {max(x_q_vals) - min(x_q_vals):.6f}')

S1: T-INDEPENDENCE TEST

  JAX [CPU (1 device(s))]: 210 branches, 115 eval pts, T=500.0 — 2.39s
T=   500  (3.7s, 115 crossings)
  Lepton:  C₀ = 5.911955  x_eff = 3.000376  dev from 3: +0.0125%
  Quark:   C₀ = 6.606742  x_eff = 1.586646

  JAX [CPU (1 device(s))]: 210 branches, 228 eval pts, T=1000.0 — 2.97s
T=  1000  (3.0s, 228 crossings)
  Lepton:  C₀ = 5.911955  x_eff = 3.000376  dev from 3: +0.0125%
  Quark:   C₀ = 6.606742  x_eff = 1.586646

  JAX [CPU (1 device(s))]: 210 branches, 458 eval pts, T=2000.0 — 5.18s
T=  2000  (5.2s, 458 crossings)
  Lepton:  C₀ = 5.911955  x_eff = 3.000376  dev from 3: +0.0125%
  Quark:   C₀ = 6.606742  x_eff = 1.586646

  JAX [CPU (1 device(s))]: 210 branches, 1143 eval pts, T=5000.0 — 11.67s
T=  5000  (11.7s, 1143 crossings)
  Lepton:  C₀ = 5.911955  x_eff = 3.000376  dev from 3: +0.0125%
  Quark:   C₀ = 6.606742  x_eff = 1.586646

  JAX [CPU (1 device(s))]: 210 branches, 2285 eval pts, T=10000.0 — 22.62s
T= 10000  (22.7s, 2285 crossings)
  Lepton:  

## S2: Window-0 CP Ratio Convergence

The window-0 CP ratio should be T-independent (NB97 #216).
If C₀ is constant and x_eff changes with T, then x_eff is NOT a real identity.
If both are constant, then m = C₀^x is exact.

In [4]:
# S2: Convergence anatomy

print('S2: WINDOW-0 CP RATIO CONVERGENCE')
print('=' * 70)
print()

print(f'{"T":>8s}  {"C₀_lep":>10s}  {"C₀_quark":>10s}  {"x_lep":>10s}  {"x_quark":>10s}  {"C₀_l³":>10s}  {"dev%":>8s}')
print('-' * 75)
for T_max in T_values:
    r = results_by_T[T_max]
    c0l3 = r['C0_l']**3
    dev = (c0l3 / target_l - 1) * 100
    print(f'{T_max:8d}  {r["C0_l"]:10.6f}  {r["C0_q"]:10.6f}  {r["x_l"]:10.6f}  {r["x_q"]:10.6f}  {c0l3:10.4f}  {dev:+8.4f}')

print()
# Check C0 stability
c0l_vals = [results_by_T[T]['C0_l'] for T in T_values]
c0q_vals = [results_by_T[T]['C0_q'] for T in T_values]
print(f'C₀_lep  spread: {max(c0l_vals) - min(c0l_vals):.8f}')
print(f'C₀_quark spread: {max(c0q_vals) - min(c0q_vals):.8f}')
print()

# If C0 changes with T, then x_eff=3 might just be an artifact of T=2000
# If C0 is stable AND x_eff is stable at 3, we have an identity
x_last = results_by_T[T_values[-1]]['x_l']
c0_last = results_by_T[T_values[-1]]['C0_l']
print(f'At T={T_values[-1]}: C₀_lep = {c0_last:.8f}, x_eff = {x_last:.8f}')
print(f'  C₀_lep³ = {c0_last**3:.6f} vs target {target_l}')
print(f'  dev from p₂: {(x_last - 3):.8f} ({(x_last/3-1)*100:.6f}%)')

S2: WINDOW-0 CP RATIO CONVERGENCE

       T      C₀_lep    C₀_quark       x_lep     x_quark       C₀_l³      dev%
---------------------------------------------------------------------------
     500    5.911955    6.606742    3.000376    1.586646    206.6299   -0.0668
    1000    5.911955    6.606742    3.000376    1.586646    206.6299   -0.0668
    2000    5.911955    6.606742    3.000376    1.586646    206.6299   -0.0668
    5000    5.911955    6.606742    3.000376    1.586646    206.6299   -0.0668
   10000    5.911955    6.606742    3.000376    1.586646    206.6299   -0.0668

C₀_lep  spread: 0.00000000
C₀_quark spread: 0.00000000

At T=10000: C₀_lep = 5.91195458, x_eff = 3.00037586
  C₀_lep³ = 206.629948 vs target 206.768
  dev from p₂: 0.00037586 (0.012529%)


## S3: All Four Mass Channels

The framework has four physical mass channels:
- QUARK_g1: m_s/m_d (intra-generation, ci=11 vs ci=191)
- LEPTON_g1: m_μ/m_e (intra-generation, ci=31 vs ci=61)
- LEPTON_g2: m_τ/m_μ (inter-generation, uses R₃ with x₃ exponent)
- QUARK_g2: m_c/m_s (inter-generation, uses multi-level)

What is the window-0 effective exponent for each?

In [6]:
# S3: All four mass channels at highest T

print('S3: ALL FOUR MASS CHANNELS — WINDOW-0 EFFECTIVE EXPONENTS')
print('=' * 70)
print()

# Use the highest T run
T_best = T_values[-1]
r = results_by_T[T_best]

print(f'Using T = {T_best}')
print()

# INTRA-GENERATION (R₃ window-0 CP → mass)
print('INTRA-GENERATION (R₃ window-0 CP → mass):')
print(f'  Lepton g1: C₀ = {r["C0_l"]:.6f}, x_eff = {r["x_l"]:.6f}')
print(f'    m_μ/m_e = C₀^x: {r["C0_l"]**r["x_l"]:.4f} (target: {target_l})')
print(f'    If x = p₂ = 3:  {r["C0_l"]**3:.4f} (dev: {(r["C0_l"]**3/target_l-1)*100:+.4f}%)')
print()
print(f'  Quark g1:  C₀ = {r["C0_q"]:.6f}, x_eff = {r["x_q"]:.6f}')
print(f'    m_s/m_d = C₀^x: {r["C0_q"]**r["x_q"]:.4f} (target: {target_q})')
print(f'    If x = p₂ = 3:  {r["C0_q"]**3:.4f} (target: {target_q}, dev: {(r["C0_q"]**3/target_q-1)*100:+.2f}%)')
print()

# INTER-GENERATION channels
target_tau_mu = 16.817  # standard value
x_tau_mu = np.log(target_tau_mu) / np.log(r['C0_l'])
print(f'INTER-GENERATION (using same C₀):')
print(f'  m_τ/m_μ: x_eff = ln({target_tau_mu})/ln({r["C0_l"]:.4f}) = {x_tau_mu:.6f}')
print(f'    NB124 formula: C₀^x₃ × p₃/p₄ (x₃ = {X3:.4f})')
print()

target_c_s = 1270.0 / 93.0  # ≈ 13.66
x_c_s = np.log(target_c_s) / np.log(r['C0_q'])
print(f'  m_c/m_s: x_eff = ln({target_c_s:.2f})/ln({r["C0_q"]:.4f}) = {x_c_s:.6f}')
print()

# Summary table
xl = r['x_l']
xq = r['x_q']
print('SUMMARY: Window-0 Effective Exponents')
print(f'{"Channel":>15s}  {"C₀":>10s}  {"x_eff":>10s}')
print('-' * 40)
print(f'{"m_μ/m_e":>15s}  {r["C0_l"]:10.6f}  {xl:10.6f}  ≈ p₂ = 3 ({(xl/3-1)*100:+.4f}%)')
print(f'{"m_s/m_d":>15s}  {r["C0_q"]:10.6f}  {xq:10.6f}')
print(f'{"m_τ/m_μ":>15s}  {r["C0_l"]:10.6f}  {x_tau_mu:10.6f}')
print(f'{"m_c/m_s":>15s}  {r["C0_q"]:10.6f}  {x_c_s:10.6f}')
print()

# The striking proximity of x_q (1.5866) and x_tau/mu (1.5883)
print(f'STRIKING: x_q = {xq:.6f} and x_τ/μ = {x_tau_mu:.6f} are nearly EQUAL')
print(f'  Difference: {x_tau_mu - xq:.6f} = {(x_tau_mu/xq-1)*100:.4f}%')
print()
print(f'  x = p₂ = 3 is SPECIFIC to the lepton intra-gen channel (m_μ/m_e).')
print(f'  Other channels cluster near x ≈ 1.59, potentially p₃/π = {p3/np.pi:.6f}')
print(f'  or ln(p₃) = {np.log(p3):.6f}')

S3: ALL FOUR MASS CHANNELS — WINDOW-0 EFFECTIVE EXPONENTS

Using T = 10000

INTRA-GENERATION (R₃ window-0 CP → mass):
  Lepton g1: C₀ = 5.911955, x_eff = 3.000376
    m_μ/m_e = C₀^x: 206.7680 (target: 206.768)
    If x = p₂ = 3:  206.6299 (dev: -0.0668%)

  Quark g1:  C₀ = 6.606742, x_eff = 1.586646
    m_s/m_d = C₀^x: 20.0000 (target: 20.0)
    If x = p₂ = 3:  288.3780 (target: 20.0, dev: +1341.89%)

INTER-GENERATION (using same C₀):
  m_τ/m_μ: x_eff = ln(16.817)/ln(5.9120) = 1.588310
    NB124 formula: C₀^x₃ × p₃/p₄ (x₃ = 1.9099)

  m_c/m_s: x_eff = ln(13.66)/ln(6.6067) = 1.384559

SUMMARY: Window-0 Effective Exponents
        Channel          C₀       x_eff
----------------------------------------
        m_μ/m_e    5.911955    3.000376  ≈ p₂ = 3 (+0.0125%)
        m_s/m_d    6.606742    1.586646
        m_τ/m_μ    5.911955    1.588310
        m_c/m_s    6.606742    1.384559

STRIKING: x_q = 1.586646 and x_τ/μ = 1.588310 are nearly EQUAL
  Difference: 0.001664 = 0.1049%

  x = p₂ = 

## S4: Algebraic Probe — Why p₂ = 3?

If x_eff(w0, lep) = 3 is exact, then:

$$m_\mu/m_e = C_0^3$$

where C₀ is the window-0 R₃ CP ratio for the lepton channel.

Can we derive this from the cascade structure? The window-0 CP is determined by
the transient and wrapping at ci=31 vs ci=61. The transient at ci is:
R₃_trans(ci) ~ exp(-κ·ci). So the CP ≈ exp(κ·(ci_g2 - ci_g1)/2) for the transient part.

If x = 3, then mass = [exp(κ·Δci/2)]³ = exp(3κ·Δci/2) where Δci = 30 = P₃.
So ln(mass) = 3·κ·P₃/2 = 3·P₃/(2√P₄) = 3×30/(2√210) = 90/(2√210).

In [7]:
# S4: Algebraic anatomy of x = 3

print('S4: ALGEBRAIC ANATOMY')
print('=' * 70)
print()

# If m_μ/m_e = C₀^3, and C₀ is the window-0 lepton CP ratio...
# The lepton crossing gap is Δci = 30 = P₃
# The pure transient CP ≈ exp(κ·Δci) = exp(P₃/√P₄)

delta_l = 30  # P₃
pure_trans_cp = np.exp(KAPPA * delta_l)
print(f'Pure transient CP = exp(κ·P₃) = exp({KAPPA:.6f}×{delta_l}) = {pure_trans_cp:.6f}')
print(f'Actual C₀ = {results_by_T[T_values[-1]]["C0_l"]:.6f}')
print(f'Ratio actual/transient = {results_by_T[T_values[-1]]["C0_l"]/pure_trans_cp:.6f}')
print()

# If x = p₂ = 3, then:
# ln(m_μ/m_e) = p₂ · ln(C₀)
# What is ln(C₀) algebraically?
lnC0 = np.log(results_by_T[T_values[-1]]['C0_l'])
print(f'ln(C₀) = {lnC0:.6f}')
print(f'p₂ · ln(C₀) = {p2 * lnC0:.6f}')
print(f'ln(m_μ/m_e) = {np.log(target_l):.6f}')
print(f'  dev = {(p2*lnC0 / np.log(target_l) - 1)*100:+.4f}%')
print()

# What is 3·ln(C₀) in terms of the primes?
# If C₀ ≈ exp(κ·P₃) = exp(P₃/√P₄), then
# 3·ln(C₀) ≈ 3·P₃/√P₄ = 3×30/√210 = 90/√210
approx = 3 * delta_l / np.sqrt(P4)
print(f'Approximation: p₂ · P₃/√P₄ = {approx:.6f}')
print(f'ln(target) = {np.log(target_l):.6f}')
print(f'Ratio = {np.log(target_l) / approx:.6f}')
print()

# The approximation misses because C₀ ≠ pure transient (wrapping affects it)
# But the KEY question: is x = p₂ exact even though C₀ requires numerical computation?
# If C₀ is whatever the cascade gives, and then mass = C₀^p₂ EXACTLY...
# That would mean the exponent is ALGEBRAIC even though the base is DYNAMICAL.

# Compare with cumulative formula:
# Cumulative: mass = CP_cum^X₄_LEP where X₄_LEP = 49/(2π)
# Window-0:   mass = C₀^3 where C₀ is window-0 CP
# The window-0 formula is dramatically simpler.

print('COMPARISON OF FORMULAS:')
print(f'  Cumulative: mass = CP_cum^[{X4_LEP:.4f}]  (X₄_LEP = γ₃/(2π) = 49/(2π))')
print(f'  Window-0:   mass = C₀^{p2}              (p₂ = chirality prime = 3)')
print()

# Are these consistent? If CP_cum^X₄_LEP = C₀^3, then:
# X₄_LEP · ln(CP_cum) = 3 · ln(C₀)
# ln(CP_cum) / ln(C₀) = 3 / X₄_LEP = 3 / (49/(2π)) = 6π/49
ratio_needed = 3 / X4_LEP
print(f'Consistency requires: ln(CP_cum)/ln(C₀) = 3/X₄_LEP = {ratio_needed:.6f}')
print(f'  = 6π/49 = {6*np.pi/49:.6f}')
print(f'  = 2πp₂/p₄² = {2*np.pi*p2/p4**2:.6f}')

S4: ALGEBRAIC ANATOMY

Pure transient CP = exp(κ·P₃) = exp(0.069007×30) = 7.926382
Actual C₀ = 5.911955
Ratio actual/transient = 0.745858

ln(C₀) = 1.776977
p₂ · ln(C₀) = 5.330930
ln(m_μ/m_e) = 5.331597
  dev = -0.0125%

Approximation: p₂ · P₃/√P₄ = 6.210590
ln(target) = 5.331597
Ratio = 0.858469

COMPARISON OF FORMULAS:
  Cumulative: mass = CP_cum^[7.7986]  (X₄_LEP = γ₃/(2π) = 49/(2π))
  Window-0:   mass = C₀^3              (p₂ = chirality prime = 3)

Consistency requires: ln(CP_cum)/ln(C₀) = 3/X₄_LEP = 0.384685
  = 6π/49 = 0.384685
  = 2πp₂/p₄² = 0.384685


## S5: The Quark Window-0 Exponent

NB133 found x_eff(w0, quark) ≈ 1.587. Is this recognizable algebraically?
Search against all simple expressions in the primes.

In [8]:
# S5: Quark window-0 exponent identification

print('S5: QUARK WINDOW-0 EXPONENT IDENTIFICATION')
print('=' * 70)
print()

x_q_best = results_by_T[T_values[-1]]['x_q']
x_l_best = results_by_T[T_values[-1]]['x_l']

print(f'Target: x_q = {x_q_best:.8f}')
print()

# Comprehensive search
candidates = {}

# Simple prime expressions
for a in range(-3, 4):
    for b in range(-3, 4):
        for c in range(-3, 4):
            for d in range(-3, 4):
                try:
                    val = p1**a * p2**b * p3**c * p4**d
                    if 0.1 < val < 100 and val != 1:
                        name = f'p1^{a}·p2^{b}·p3^{c}·p4^{d}'
                        candidates[name] = val
                except:
                    pass

# Expressions with π, √primes, φ, λ
extra = {
    'p₃/π': p3/np.pi,
    'φ(P₃)/p₃': 8/p3,
    'π/2': np.pi/2,
    'ln(P₃)': np.log(30),
    'ln(P₄)/ln(p₃)': np.log(210)/np.log(5),
    'p₂/√(p₃-1)': p2/np.sqrt(p3-1),
    '√(p₃/2)': np.sqrt(p3/2),
    'p₃/√(P₃/p₂)': p3/np.sqrt(30/3),
    'x_l/x_q (ratio)': x_l_best/x_q_best,
    'p₂/x_q': p2/x_q_best,
    'X₃/X₂': X3/X2,
    'λ(p₄)/ω(P₄)': 6/4,
    'ln(p₄²)': np.log(49),
    'ln(P₄)/p₂': np.log(210)/3,
    'P₃/(2π·p₂)': 30/(2*np.pi*3),
    'p₂·P₃/P₄': 3*30/210,
    'x_l/2': x_l_best/2,
}
candidates.update(extra)

# Sort by closeness to x_q
ranked = sorted(candidates.items(), key=lambda kv: abs(kv[1] - x_q_best))

print(f'Top 15 candidates (sorted by |deviation|):')
for i, (name, val) in enumerate(ranked[:15]):
    dev = (val / x_q_best - 1) * 100
    print(f'  {i+1:2d}. {name:>30s} = {val:10.6f}  dev = {dev:+8.4f}%')

print()
# The ratio x_l / x_q should be meaningful
ratio = x_l_best / x_q_best
print(f'Lepton/Quark exponent ratio: x_l/x_q = {ratio:.6f}')
print(f'  Nearby: 2·p₂/p₂ = 2, but ratio = {ratio:.4f}')
print(f'  Nearby: p₂/x_q itself...')
print(f'  p₂·ln(C₀_q)/ln(C₀_l) = {p2 * np.log(results_by_T[T_values[-1]]["C0_q"]) / np.log(results_by_T[T_values[-1]]["C0_l"]):.6f}')

S5: QUARK WINDOW-0 EXPONENT IDENTIFICATION

Target: x_q = 1.58664640

Top 15 candidates (sorted by |deviation|):
   1.          p1^2·p2^-2·p3^2·p4^-1 =   1.587302  dev =  +0.0413%
   2.          p1^-3·p2^-3·p3^0·p4^3 =   1.587963  dev =  +0.0830%
   3.                           p₃/π =   1.591549  dev =  +0.3090%
   4.                     P₃/(2π·p₂) =   1.591549  dev =  +0.3090%
   5.                        √(p₃/2) =   1.581139  dev =  -0.3471%
   6.                    p₃/√(P₃/p₂) =   1.581139  dev =  -0.3471%
   7.          p1^-3·p2^2·p3^-1·p4^1 =   1.575000  dev =  -0.7340%
   8.           p1^2·p2^3·p3^1·p4^-3 =   1.574344  dev =  -0.7754%
   9.           p1^3·p2^0·p3^-1·p4^0 =   1.600000  dev =  +0.8416%
  10.                       φ(P₃)/p₃ =   1.600000  dev =  +0.8416%
  11.                            π/2 =   1.570796  dev =  -0.9990%
  12.           p1^2·p2^0·p3^-3·p4^2 =   1.568000  dev =  -1.1752%
  13.          p1^-2·p2^2·p3^1·p4^-1 =   1.607143  dev =  +1.2918%
  14.           

## S6: Synthesis and Scorecard

In [12]:
# S6: Residual anatomy — CORRECTED

print('S6: RESIDUAL ANATOMY — Cumulative vs Window-0')
print('=' * 70)
print()

# === Step 1: The cumulative pipeline is T-dependent ===
# The established pipeline (accumulate_sectors + cp_pair_ratios + mass_ratios)
# averages sector RMS across ALL crossings. At large T, late-time crossings
# where the CP ratio → 1 (steady-state) dilute the early-time signal.
# This is EXACTLY why NB97 introduced window-0 concentration.

# Demonstrate T-dependence of cumulative pipeline
print('CUMULATIVE PIPELINE T-DEPENDENCE:')
print(f'{"T":>8s}  {"R₄_cum":>10s}  {"R₄^X₄_LEP":>12s}  {"dev from target":>16s}')
print('-' * 52)

for T_val in T_values:
    ci_t = SA.coprime_indices(T_val)
    t_ev = ci_t.astype(float)
    res_t = integrate_all_branches_jax(branches, t_ev, float(T_val),
                                        kappa=KAPPA, epsilon=EPSILON, omega=OMEGA)
    a3_t, a5_t, a7_t = SA.sector_labels(ci_t)
    srms = SolenoidSystem.accumulate_sectors(res_t, ci_t, a3_t, a5_t, a7_t)
    cpr = SolenoidSystem.cp_pair_ratios(srms)
    R4_l = cpr['LEPTON'][3]
    mass_t = R4_l ** X4_LEP
    dev = (mass_t / target_l - 1) * 100
    print(f'{T_val:8d}  {R4_l:10.6f}  {mass_t:12.4f}  {dev:+16.4f}%')

print()
print('  The cumulative R₄ DECREASES with T as dilution grows.')
print('  NB97 showed this: all CP asymmetry concentrates in window 0.')
print()

# === Step 2: Window-0 formula is T-independent ===
C0_l_val = results_by_T[T_values[-1]]['C0_l']
x_final_l = results_by_T[T_values[-1]]['x_l']
mass_w0 = C0_l_val ** 3

print('WINDOW-0 FORMULA (T-independent):')
print(f'  C₀ = {C0_l_val:.8f} (identical at all T)')
print(f'  C₀^p₂ = {mass_w0:.4f} (dev: {(mass_w0/target_l-1)*100:+.4f}%)')
print()

# === Step 3: Algebraic relationship between exponents ===
# The two formulas use different bases but related exponents:
#   Cumulative: mass = R₄_cum^X₄_LEP (where X₄_LEP = p₄²/(2π))
#   Window-0:   mass = C₀^p₂
# These are NOT equivalent formulas — the cumulative one is T-dependent.
# But the EXPONENT RATIO is purely algebraic:
#   X₄_LEP / p₂ = [p₄²/(2π)] / p₂ = p₄² / (2π·p₂) = 49/(6π)

ratio = X4_LEP / p2
exact = p4**2 / (2 * np.pi * p2)
print(f'EXPONENT RELATIONSHIP:')
print(f'  X₄_LEP / p₂ = {ratio:.6f}')
print(f'  p₄² / (2π·p₂) = 49/(6π) = {exact:.6f}')
print(f'  Match: {np.isclose(ratio, exact)}')
print()
print(f'  The cumulative exponent X₄_LEP = {X4_LEP:.4f} is the window-0')
print(f'  exponent p₂ = 3 amplified by p₄²/(2π·p₂) = 49/(6π) ≈ 2.60')
print(f'  to compensate for the dilution that compresses R₄_cum < C₀.')
print()

# === Step 4: Is x = 3 exact? ===
residual_x = x_final_l - 3
c0_needed = target_l ** (1.0/3)
print(f'IS x = p₂ EXACT?')
print(f'  x_eff = {x_final_l:.8f}')
print(f'  p₂    = 3')
print(f'  δx    = {residual_x:.8f} ({(residual_x/3)*100:.4f}%)')
print(f'  C₀^δx correction:  {C0_l_val**residual_x:.8f} ({(C0_l_val**residual_x - 1)*100:+.4f}%)')
print(f'  C₀ for exact x=3:  {c0_needed:.8f}')
print(f'  C₀ actual:         {C0_l_val:.8f}')
print(f'  C₀ deficit:        {(C0_l_val/c0_needed - 1)*100:+.4f}%')
print()
print(f'  The 0.013% deviation of x from 3 accounts for the entire')
print(f'  0.067% mass deviation. Whether x = 3 exactly (with C₀')
print(f'  carrying a small correction) or x = 3 + ε cannot be')
print(f'  determined numerically — requires analytic derivation.')

S6: RESIDUAL ANATOMY — Cumulative vs Window-0

CUMULATIVE PIPELINE T-DEPENDENCE:
       T      R₄_cum     R₄^X₄_LEP   dev from target
----------------------------------------------------
  JAX [CPU (1 device(s))]: 210 branches, 115 eval pts, T=500.0 — 1.83s
     500    3.993660    48960.7264       +23579.0637%
  JAX [CPU (1 device(s))]: 210 branches, 228 eval pts, T=1000.0 — 2.76s
    1000    3.253433     9897.9597        +4686.9882%
  JAX [CPU (1 device(s))]: 210 branches, 458 eval pts, T=2000.0 — 4.96s
    2000    2.460147     1119.2987         +441.3307%
  JAX [CPU (1 device(s))]: 210 branches, 1143 eval pts, T=5000.0 — 11.64s
    5000    1.781547       90.3370          -56.3100%
  JAX [CPU (1 device(s))]: 210 branches, 2285 eval pts, T=10000.0 — 22.74s
   10000    1.449064       18.0408          -91.2748%

  The cumulative R₄ DECREASES with T as dilution grows.
  NB97 showed this: all CP asymmetry concentrates in window 0.

WINDOW-0 FORMULA (T-independent):
  C₀ = 5.91195458 (ident

In [14]:
# ── Scorecard ──

print('NB134 SCORECARD')
print('=' * 70)
print()

print('INVESTIGATION: Is the window-0 effective exponent x = p₂ = 3 a real identity?')
print()

print('CONFIRMED OBSERVATIONS:')
print()
print('  1. T-INDEPENDENCE: x_eff(w0, lepton) = 3.000376 is identical')
print('     at T = 500, 1000, 2000, 5000, 10000 (spread = 0.000000)')
print()
print('  2. C₀ T-INDEPENDENCE: Window-0 CP ratio = 5.911955 is identical')
print('     across all T values (window-0 concentration, NB97 #216)')
print()
print('  3. MASS PREDICTION: C₀³ = 206.630 vs m_μ/m_e = 206.768 (-0.067%)')
print()
print('  4. CHANNEL SPECIFICITY: x = p₂ appears ONLY in the lepton intra-gen')
print('     channel. Quark g1: x = 1.587, tau/mu: x = 1.588, m_c/m_s: x = 1.385')
print()
print('  5. QUARK-TAU PROXIMITY: x(quark g1) ~ x(tau/mu) ~ 1.587-1.588')
print('     Both close to p₃/pi = 1.592 (0.3%) -- less clean than lepton hit')
print()
print('  6. RESIDUAL: 0.013% deviation of x from 3 accounts for the')
print('     entire 0.067% mass deviation via C₀^dx = 1.00067')
print()
print('  7. CUMULATIVE T-DEPENDENCE: The cumulative pipeline (accumulate_sectors)')
print('     is massively T-dependent: R₄^X₄_LEP gives 48961 at T=500,')
print('     1119 at T=2000, 18 at T=10000. There is no stable T.')
print('     Window-0 avoids this entirely.')
print()
print('  8. EXPONENT RATIO: X₄_LEP / p₂ = p₄²/(2π·p₂) = 49/(6π) (EXACT)')
print('     The cumulative exponent is the window-0 exponent amplified')
print('     by this factor to compensate for dilution.')
print()

# Identity assessment
x_final = results_by_T[T_values[-1]]['x_l']
c0_final = results_by_T[T_values[-1]]['C0_l']
print('IDENTITY ASSESSMENT:')
print()
print('  #277 (PROVISIONAL): m_μ/m_e = C₀_lep^p₂')
print('    where C₀ is the window-0 R₃ CP ratio')
print(f'    Value: {c0_final:.6f}³ = {c0_final**3:.4f}')
print(f'    Target: {target_l}')
print(f'    Deviation: -0.067%')
print(f'    T-independent: YES')
print(f'    Verdict: PROVISIONAL -- x = {x_final:.6f}, not exactly 3.')
print(f'             0.013% deviation may be structural or residual.')
print(f'             Analytic derivation needed.')
print()
print(f'    NOTE: C₀ is dynamical; p₂ is algebraic.')
print(f'    The EXPONENT is algebraic even though the BASE is dynamical.')
print()
print(f'    Correspondential: p₂ = 3 = chirality prime = three discrete degrees.')
print(f'    Mass = CP asymmetry received through celestial/spiritual/natural.')
print()
print(f'Running total: 277 predictions/identities, 0 free parameters')
print(f'  (276 established + 1 provisional)')

NB134 SCORECARD

INVESTIGATION: Is the window-0 effective exponent x = p₂ = 3 a real identity?

CONFIRMED OBSERVATIONS:

  1. T-INDEPENDENCE: x_eff(w0, lepton) = 3.000376 is identical
     at T = 500, 1000, 2000, 5000, 10000 (spread = 0.000000)

  2. C₀ T-INDEPENDENCE: Window-0 CP ratio = 5.911955 is identical
     across all T values (window-0 concentration, NB97 #216)

  3. MASS PREDICTION: C₀³ = 206.630 vs m_μ/m_e = 206.768 (-0.067%)

  4. CHANNEL SPECIFICITY: x = p₂ appears ONLY in the lepton intra-gen
     channel. Quark g1: x = 1.587, tau/mu: x = 1.588, m_c/m_s: x = 1.385

  5. QUARK-TAU PROXIMITY: x(quark g1) ~ x(tau/mu) ~ 1.587-1.588
     Both close to p₃/pi = 1.592 (0.3%) -- less clean than lepton hit

  6. RESIDUAL: 0.013% deviation of x from 3 accounts for the
     entire 0.067% mass deviation via C₀^dx = 1.00067

  7. CUMULATIVE T-DEPENDENCE: The cumulative pipeline (accumulate_sectors)
     is massively T-dependent: R₄^X₄_LEP gives 48961 at T=500,
     1119 at T=2000, 18 a